# Fetch a Circuit entity from staging

Fetches the Circuit entity `a5d839e1-3a65-4237-a2b4-41710edfab0a` from the
**staging** entitycore database via `entitysdk` and prints its metadata —
particularly its `scale`.

Requires staging access: `obi_auth.get_token(environment="staging")` opens a
browser for authentication on first use.

## Connect to the staging entitycore database

In [ ]:
import obi_one as obi
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from obi_auth.config import settings as obi_auth_settings
from obi_auth.storage import Storage as ObiAuthStorage
from obi_auth.typedef import DeploymentEnvironment

# obi_auth caches the last sign-in on disk
# (~/.config/obi-auth/token_staging.json, Fernet-encrypted) and get_token()
# silently reuses it. Clear it first so a *different* signed-in account
# re-authenticates instead of reusing the previous account's token.
ObiAuthStorage(
    config_dir=obi_auth_settings.config_dir,
    environment=DeploymentEnvironment.staging,
).clear()

token = get_token(environment='staging')
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url='https://staging.openbraininstitute.org/api/entitycore',
    project_context=project_context,
    token_manager=token,
)
print('connected to staging entitycore')

## Fetch the circuit and print its `scale`

In [ ]:
CIRCUIT_ID = 'dc57f86d-a591-4d7e-a65d-54e097e8a021' # FlyWire Test - Shared results + tests

circuit = db_client.get_entity(entity_id=CIRCUIT_ID, entity_type=models.Circuit)

print(f'name:  {circuit.name}')
print(f'scale: {circuit.scale}')

EntitySDKError: HTTP error 403 for GET https://staging.openbraininstitute.org/api/entitycore/circuit/39760158-2183-4ffc-84d2-595ada423932
data       : None
json       : null
params     : None
response   : {"error_code":"NOT_AUTHORIZED","message":"User not authorized for the given virtual-lab-id or project-id","details":null}

## Full metadata

In [ ]:
key_fields = [
    'id', 'name', 'description', 'scale', 'build_category',
    'number_neurons', 'number_synapses', 'number_connections',
    'has_morphologies', 'has_point_neurons', 'has_spines',
    'has_electrical_cell_models', 'target_simulator',
    'brain_region', 'subject', 'root_circuit_id', 'atlas_id',
    'creation_date', 'update_date',
]
for f in key_fields:
    print(f'{f:>26}: {getattr(circuit, f, None)}')

print('\n--- full metadata (JSON, assets excluded) ---')
print(circuit.model_dump_json(indent=2, exclude={'assets'}))